# 13b · More training data, with the legal moves in the prompt (`sft_100k_legal_moves`)

**Why this run.** Two changes to SFT improved the model, each on its own:
- **more data** (notebook 12): `sft_100k`, trained on 100,000 positions instead of 20,000, is clearly better than `sft_v1` on held-out game positions (more legal answers, more of Stockfish's moves, about 40 centipawns less lost per move), and solves 9.4% of the puzzles vs. 7.8%;
- **the legal moves in the prompt** (notebook 13): `sft_legal_moves` is our best model on puzzles (13.2%), never answers illegally, and is the first model to clearly beat `sft_v1`. But on game positions, its moves that are not Stockfish's lose as much as random moves.

Nobody has trained a model with both. This notebook asks: **do the two gains add up?** More data improved exactly what the legal-move model lacks (move quality in ordinary positions), so the combination could be our strongest SFT model, and a better starting point for DPO and RL (notebook 15). If the gains do not add up, that is also a result for the paper.

**What changes, and what stays the same.** Compared with `sft_legal_moves`, only the amount of data; compared with `sft_100k`, only the prompt:

| | `sft_100k` (notebook 12) | `sft_legal_moves` (notebook 13) | `sft_100k_legal_moves` (this notebook) |
|---|---|---|---|
| Training positions | 100,000 | 20,000 | **100,000** (the first 20,000 are `sft_v1`'s, in the same order) |
| Legal moves listed in the prompt | no | yes | **yes** |
| Answer the model learns | the move the player made | the same | the same |
| Model, LoRA, learning rate, epochs (1), seed | `configs/sft.yaml` | the same | the same |

As in notebook 12, a copy of the model is saved after 20k, 40k, 60k and 80k examples and graded. The same caveat applies: the copies show progress during one run, not what a model trained on fewer examples would score, because the learning rate is still high when the early copies are saved.

**Running it on Databricks**
1. Open this notebook in the shared Git folder and pick **Serverless GPU → Small (1×A10)**. Plain *Serverless* has no GPU. Medium and Large cost much more and are not needed here.
2. Set `QUICK_CHECK = True` in the Settings cell and click **Run all**. This takes a few minutes, uses tiny sizes, and saves nothing to `results/`. It catches setup problems before the long run. (The first cell restarts Python once; if Run all stops there, click Run all again.)
3. Set `QUICK_CHECK = False` and run the real thing as a **Job**, so the browser can close: New → Job → type *Notebook* → this notebook → compute *Serverless GPU, 1×A10*; add an email notification. Separate Jobs can run at the same time.
4. Afterwards, commit and push **only `results/`** from the Git folder. Leave `notebooks/` unticked: Databricks cannot save notebook outputs to git, so committing a notebook from there would erase its outputs on GitHub. The outputs of a Job run are on the Job's run page; every figure and table is also saved in `results/figures/`.

**Time:** about 6 hours of training on an A10 (the prompts with the list are about 2.4 times longer than in notebook 12) and about 30 minutes of grading, roughly $12–14 of credits. If the Job stops early, the copies saved so far are kept in `checkpoints/` and can still be graded.

**Outputs**
- the adapter in `checkpoints/sft_100k_legal_moves/` and the copies in `checkpoints/sft_100k_legal_moves_at_20k/`, `_at_40k/`, `_at_60k/`, `_at_80k/` (not on GitHub)
- results: `results/sft_100k_legal_moves_training.json`, and for the final model and each copy `results/<name>.json`, `results/<name>_dev.json`, and rows in `results/summary.csv`
- every figure and table of this experiment in `results/figures/sft_100k_legal_moves/`

In [ ]:
# On Databricks: install our packages and Stockfish for this session. Does nothing on other machines.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))  # so Databricks finds our code in src/
from chesslm.databricks import setup_databricks

setup_databricks()

In [ ]:
# Settings
RUN_NAME = "sft_100k_legal_moves"
QUICK_CHECK = False   # True = a few-minute run with tiny sizes, to check that everything works. Saves nothing to results/.
TRAIN_EXAMPLES = 100_000                    # as in sft_100k (sft_legal_moves: 20,000)
SAVE_AT = [20_000, 40_000, 60_000, 80_000]  # save and grade a copy after this many examples

import gc
import math

import pandas as pd
import torch
from IPython.display import display

pd.set_option("display.width", 200)  # print wide tables on one line

from chesslm import experiments, plots
from chesslm.config import REPO_ROOT, load_config
from chesslm.evaluate import load_eval_model
from chesslm.experiments import compare_with, figures_dir, grade_puzzles, puzzle_table, save_table, save_training
from chesslm.model import load_tokenizer
from chesslm.prompts import build_messages
from chesslm.train_sft import copy_dir, examples_label, sample_examples, to_dataset, train

sft_cfg = load_config("sft")
prompt_cfg, eval_cfg = load_config("prompt"), load_config("eval")
prompt_cfg["show_legal_moves"] = True       # as in sft_legal_moves
configs = {"model": sft_cfg["model"], "prompt": prompt_cfg, "eval": eval_cfg}
SEED = sft_cfg["training"]["seed"]
SFT_V1_EXAMPLES = sft_cfg["data"]["train_examples"]          # 20,000
sft_cfg["data"]["train_examples"] = TRAIN_EXAMPLES
sft_cfg["training"]["eval_steps"] = 500                      # validation loss less often, to save time (as in 12)

if QUICK_CHECK:
    experiments.use_quick_check_folder()
    RUN_NAME += "_check"
    sft_cfg["data"].update(trial_examples=16, train_examples=96, val_examples=16)
    sft_cfg["training"].update(logging_steps=1, eval_steps=1)
    SAVE_AT = [32, 64]
PUZZLE_LIMIT = 5 if QUICK_CHECK else None
ADAPTER_DIR = REPO_ROOT / "checkpoints" / RUN_NAME
FIGURES = figures_dir(RUN_NAME)
print(f"Run: {RUN_NAME} | {sft_cfg['data']['train_examples']:,} training examples | legal moves in the prompt: "
      f"{prompt_cfg['show_legal_moves']} | copies after {SAVE_AT} | figures and tables -> {FIGURES}")

## 1. Training data and prompt length

The same seeded sample of 100,000 positions as `sft_100k` (the first 20,000 are `sft_v1`'s and `sft_legal_moves`' positions, in the same order; the cell checks this), and the same 1,000 validation positions.

The list makes prompts longer. Training cuts every example to a maximum length, and an example cut short would lose its answer, so the cell measures the longest example and sets the maximum length above it (as in notebook 13).

In [ ]:
n = sft_cfg["data"]
train_positions = pd.read_parquet(REPO_ROOT / "data" / "processed" / "train.parquet")
train_rows = sample_examples(train_positions, n["train_examples"], SEED)
first_rows = sample_examples(train_positions, SFT_V1_EXAMPLES, SEED)
same_start = list(train_rows.fen[:SFT_V1_EXAMPLES]) == list(first_rows.fen) if not QUICK_CHECK else True
assert same_start, "the first 20,000 positions should be sft_v1's and sft_legal_moves'"
val_rows = sample_examples(pd.read_parquet(REPO_ROOT / "data" / "processed" / "val.parquet"), n["val_examples"], SEED)
print("One training prompt:\n" + build_messages(train_rows.iloc[0].to_dict(), prompt_cfg)[0]["content"] + "\n")

tokenizer = load_tokenizer(sft_cfg["model"])
def length(row):
    messages = build_messages(row, prompt_cfg) + [{"role": "assistant", "content": row[f"move_{prompt_cfg['notation']}"]}]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return len(tokenizer(text, add_special_tokens=False).input_ids)
lengths = pd.Series([length(r) for r in pd.concat([train_rows, val_rows]).to_dict("records")])
sft_cfg["training"]["max_seq_length"] = max(256, math.ceil(lengths.max() / 64) * 64)
del tokenizer

data_table = pd.DataFrame({"Training positions": [SFT_V1_EXAMPLES, len(train_rows)],
                           "Average length (tokens)": [None, round(lengths.mean())],
                           "Longest (tokens)": [None, lengths.max()]},
                          index=pd.Index(["sft_legal_moves", RUN_NAME], name="Model"))
save_table(data_table, FIGURES, "training_data",
           "Training positions per model, and the length of a training example (prompt with the legal moves, and "
           "answer) in tokens. The first 20,000 positions of this run are exactly sft_legal_moves'.")
display(data_table)
print(f"Maximum training length set to {sft_cfg['training']['max_seq_length']} tokens, so no example is cut short.")

## 2. Trial run

A short run on the first few hundred training examples, to check that the loss goes down on this machine before the long run.

In [ ]:
trial_ds = to_dataset(train_rows.head(n["trial_examples"]), prompt_cfg)
val_ds = to_dataset(val_rows, prompt_cfg)
trial_log, trial_compute = train(trial_ds, val_ds.select(range(min(100, len(val_ds)))), sft_cfg,
                                 REPO_ROOT / "checkpoints" / f"{RUN_NAME}_trial")
losses = trial_log.loss.dropna()
print(f"Training loss: {losses.iloc[0]:.3f} at the start -> {losses.iloc[-1]:.3f} at the end")
gc.collect()
torch.cuda.empty_cache()

## 3. Full training run, saving copies along the way

The figure shows the loss on training examples and on the 1,000 held-out validation positions.

In [ ]:
train_ds = to_dataset(train_rows, prompt_cfg)
print(f"train: {len(train_ds):,} examples  validation: {len(val_ds):,} examples")
log, compute = train(train_ds, val_ds, sft_cfg, ADAPTER_DIR, save_at=SAVE_AT)
save_training(RUN_NAME, {"sft": sft_cfg, "prompt": prompt_cfg, "save_at": SAVE_AT}, compute, log)
plots.loss_curve(log, FIGURES / "training_loss.png", title=f"{RUN_NAME}: loss during training");

## 4. Grade every copy and the final model on the puzzles

The same grader, puzzles, and greedy decoding as every other model, with the legal moves in the prompt: the 200 dev puzzles first, then the 1,000 test puzzles. Copies that are missing (for example, if training stopped early) are skipped.

In [ ]:
models = {f"{RUN_NAME}_at_{examples_label(k)}": (k, copy_dir(ADAPTER_DIR, k)) for k in SAVE_AT}
models[RUN_NAME] = (n["train_examples"], ADAPTER_DIR)
for name, (examples, adapter) in models.items():
    if not adapter.exists():
        print(f"{name}: skipped, {adapter} not found")
        continue
    gc.collect()
    torch.cuda.empty_cache()
    tokenizer, model = load_eval_model(sft_cfg["model"], adapter_path=adapter)
    grade_puzzles(model, tokenizer, "dev", f"{name}_dev", configs, str(adapter), PUZZLE_LIMIT)
    grade_puzzles(model, tokenizer, "test", name, configs, str(adapter), PUZZLE_LIMIT)
    del model

## 5. Results: do the two gains add up?

**How to read this section**
- **The 2 × 2 table** gives the puzzle accuracy of the four models that differ in data size and prompt: `sft_v1` (20k, no list), `sft_100k` (100k, no list), `sft_legal_moves` (20k, list), and this run (100k, list). If the gains add up, this run is clearly above `sft_legal_moves`.
- **The paired comparisons** count puzzles solved by only one of two models; a p-value below 0.05 means the difference is unlikely to be chance. This run vs. `sft_legal_moves` is the effect of more data with the list; vs. `sft_100k`, the effect of the list with more data.
- **Accuracy when legal** is puzzle accuracy among the legal answers only. Both legal-move models should be close to 100% legal, so for them it is about the same as puzzle accuracy.
- **The scaling curves** put this run's copies next to `sft_100k`'s. Read them as progress during one run (see the caveat at the top).

All numbers are on the 1,000 test puzzles; a random legal move solves 5.6%.

In [ ]:
graded = set(pd.read_csv(experiments.RESULTS / "summary.csv").run)


def scaling_table(run):
    """Test-puzzle results of a run's saved copies and final model, indexed by training examples seen (None if not graded)."""
    names = {f"{run}_at_{examples_label(k)}": k for k in SAVE_AT} | {run: n["train_examples"]}
    names = {name: k for name, k in names.items() if name in graded}
    if not names:
        return None
    table = puzzle_table(list(names)).reset_index()
    table.insert(1, "Training examples", list(names.values()))
    return table.set_index("Training examples")[["Model", "Puzzle accuracy (%)", "Legal (%)", "ACPL (cp)"]]


scaling = scaling_table(RUN_NAME)
save_table(scaling, FIGURES, "scaling",
           "Test-puzzle results of the copies saved during training and of the final model. ACPL is over legal answers only.")
display(scaling)

accuracy = lambda run: puzzle_table([run]).iloc[0]["Puzzle accuracy (%)"] if run in graded else None
two_by_two = pd.DataFrame({"20,000 positions": [accuracy("sft_v1"), accuracy("sft_legal_moves")],
                           "100,000 positions": [accuracy("sft_100k"), accuracy(RUN_NAME)]},
                          index=pd.Index(["No list", "Legal moves listed"], name="Prompt"))
save_table(two_by_two, FIGURES, "two_by_two",
           "Test-puzzle accuracy (%) of the four SFT models: sft_v1, sft_100k (top row); sft_legal_moves, "
           "sft_100k_legal_moves (bottom row). Empty = not graded.")
display(two_by_two)

panels = [("Puzzle accuracy (%)", "puzzle accuracy (%)"), ("Legal (%)", "legal answers (%)")]
curves = {f"legal moves listed ({RUN_NAME})": scaling}
no_list = scaling_table("sft_100k")
if no_list is not None:
    curves = {"no list (sft_100k)": no_list, **curves}
plots.scaling_curves(curves, panels, {"random legal move": {"Puzzle accuracy (%)": 5.6}},
                     FIGURES / "scaling_curves.png", title="More data, with and without the legal moves in the prompt");

In [ ]:
runs = [run for run in ["sft_v1", "sft_100k", "sft_legal_moves", RUN_NAME] if run in graded]
table = puzzle_table(runs, FIGURES)
display(table)

choice = table[["Puzzle accuracy (%)", "Legal (%)"]].copy()
choice["Accuracy when legal (%)"] = (choice["Puzzle accuracy (%)"] / choice["Legal (%)"] * 100).round(1)
save_table(choice, FIGURES, "accuracy_when_legal",
           "Accuracy when legal = puzzle accuracy among the legal answers only (accuracy / legal rate).")
display(choice)

pairs = [(RUN_NAME, other) for other in ["sft_legal_moves", "sft_100k", "sft_v1"] if other in graded]
first_copy = f"{RUN_NAME}_at_{examples_label(SAVE_AT[0])}"
if first_copy in graded and "sft_legal_moves" in graded:
    pairs.append((first_copy, "sft_legal_moves"))
comparison = pd.DataFrame([compare_with(new, old) for new, old in pairs]).set_index("model")
save_table(comparison, FIGURES, "paired_comparison",
           "Puzzles solved by only one of the two models, and the sign-test p-value (below 0.05 = unlikely to be chance).")
display(comparison)

In [ ]:
summary = pd.read_csv(experiments.RESULTS / "summary.csv")
plots.compare_metrics(summary, runs, FIGURES / "puzzle_metrics.png",
                      title="More data and the legal-move list, separately and together");
plots.accuracy_by_band(summary, runs, FIGURES / "accuracy_by_band.png",
                       title="Puzzle accuracy by puzzle difficulty (rating band)");
plots.outcome_breakdown(summary, runs, FIGURES / "answer_outcomes.png", title="Are the answers legal moves?");

## Summary of outputs

- `results/figures/sft_100k_legal_moves/`: `training_data` (table), `training_loss.png`, `scaling` (table), `two_by_two` (table), `scaling_curves.png`, `puzzle_results` (table), `accuracy_when_legal` (table), `paired_comparison` (table), `puzzle_metrics.png`, `accuracy_by_band.png`, `answer_outcomes.png`
- `results/sft_100k_legal_moves.json` and `results/sft_100k_legal_moves_at_<N>k.json`: every test answer and its grade; `results/sft_100k_legal_moves_training.json`: settings, compute, training log
- The final model is also graded on held-out game positions and in full games in notebook 11, with the legal moves in the prompt.
- If it clearly beats `sft_legal_moves`, the SFT → RL combination of notebook 15 is run again from it.